# Pensieve ML - Phase 5: Grounded Reflection Generation

> **ETHICAL AND SCIENTIFIC NOTICE**:
> - Phase 5 performs **observational reflection synthesis**, not clinical diagnosis or medical assessment.
> - Language generation operates under a **two-stage safety architecture**: pre-generation policy gating and post-generation safety validation.
> - Reflections are grounded strictly in concepts retrieved by Phase 4; outside theories are forbidden.
>
> **DATASET NOTICE**:
> The grounding concepts reference the representative **20-concept DEVELOPMENT / TEST dataset**. This is NOT the final 54-concept production set.

In [ ]:
import sys
import os

# Ensure project root is on sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from ml.reflection import (
    ReflectionGenerator,
    ReflectionInput,
    ReflectionPolicy,
    ReflectionValidator,
    build_reflection_prompt,
    MockLLMClient,
)
from ml.reflection.evaluate import evaluate_reflection_pipeline

print("Phase 5 reflection modules imported successfully.")

## 1. Constructing the ReflectionInput Contract
Bridges structured signals from Phases 1–3 and grounded concepts from Phase 4 without sending raw journal databases.

In [ ]:
sample_input = ReflectionInput(
    data_summary={"entry_count": 12, "span_days": 28, "num_windows": 4},
    emotion_patterns=[
        "Annoyance elevated during early weeks; joy and gratitude increasing later.",
        "Nervousness associated with product release milestones."
    ],
    theme_patterns=["Work & Engineering (50%)", "Health & Fitness (30%)"],
    linguistic_patterns={"negation_ratio": 0.05, "question_count": 2},
    temporal_patterns=["Transition from work deadline stress to active health habits."],
    recurring_patterns=["Frequent term: 'deadline'", "Frequent term: 'running'"],
    retrieved_concepts=[
        {
            "concept_id": "stress_appraisal_framework",
            "name": "Transactional Stress Appraisal",
            "category": "cognitive_reflective",
            "definition": "A model balancing perceived demands against recognized coping options.",
            "explanation": "Helps reflect on workload and recovery balance.",
            "source": "Lazarus, R. S., & Folkman, S. (1984). Stress, Appraisal, and Coping.",
            "cautions": ["Descriptive theoretical model; not a clinical burnout diagnosis."],
            "similarity": 0.4606,
        }
    ],
)

print(f"Created ReflectionInput with {sample_input.data_summary['entry_count']} entries across {sample_input.data_summary['span_days']} days.")

## 2. Pre-Generation Policy Evaluation
Evaluates minimum data safeguards and rolling rate limits before any model generation.

In [ ]:
policy = ReflectionPolicy()
policy_check = policy.evaluate(
    entry_count=sample_input.data_summary["entry_count"],
    span_days=sample_input.data_summary["span_days"],
    retrieved_concepts=sample_input.retrieved_concepts,
)

print(f"Policy Allowed: {policy_check.allowed}")
print(f"Policy Message: {policy_check.message}")

## 3. Prompt Construction & Grounding Inspection

In [ ]:
prompts = build_reflection_prompt(sample_input)
print("--- System Instruction Preview (first 250 chars) ---")
print(prompts["system"][:250] + "...\n")
print("--- User Prompt Payload ---")
print(prompts["user"])

## 4. Generating the Grounded Reflection
Executes provider-isolated generation, confidence capping (max 0.80), and post-generation safety validation.

In [ ]:
generator = ReflectionGenerator()
result = generator.generate_reflection(sample_input)

print(f"Generation Status: {result['status'].upper()}")
print(f"Confidence:        {result['confidence']} (Capped <= 0.80)")
print(f"Grounded Concept:  {result['grounded_concepts'][0]['name']}")
print(f"Source Cited:      {result['grounded_concepts'][0]['source']}\n")
print("Reflection Text:")
print(result["reflection"])
print(f"\nMandatory Disclaimer:\n{result['disclaimer']}")

## 5. Demonstrating Safety and Policy Rejections
Demonstrating that unsafe outputs or invalid requests are safely caught without bypassing the safety layer.

In [ ]:
# Rejection A: Sparse data (< 3 entries)
sparse_res = generator.generate_reflection({
    "data_summary": {"entry_count": 2, "span_days": 10},
    "retrieved_concepts": [{"concept_id": "stoic_dichotomy_of_control"}],
})
print(f"Sparse Data Status:  {sparse_res['status'].upper()} (Reason: {sparse_res.get('reason')})")

# Rejection B: Unsafe Diagnostic Output caught by Validator
unsafe_client = MockLLMClient(canned_response={
    "reflection": "You have clinical depression and anxiety.",
    "grounded_concepts": [{"concept_id": "stress_appraisal_framework"}],
    "confidence": 0.95,
    "disclaimer": "This reflection describes observable patterns in journal text for personal contemplation. It does not constitute psychological, psychiatric, or medical advice or diagnosis.",
})
unsafe_gen = ReflectionGenerator(client=unsafe_client)
unsafe_res = unsafe_gen.generate_reflection(sample_input)
print(f"Diagnostic Status:   {unsafe_res['status'].upper()} (Reason: {unsafe_res.get('reason')})")
print(f"Validation Errors:   {unsafe_res.get('validation_errors')}")

## 6. Running the 12-Scenario Reflection Benchmark

In [ ]:
eval_data = evaluate_reflection_pipeline()
summary = eval_data["summary_metrics"]

print("BENCHMARK SUMMARY METRICS:")
print(f"  Total Test Scenarios          : {summary['total_benchmark_cases']}")
print(f"  Passed Cases                  : {summary['passed_cases']} / {summary['total_benchmark_cases']}")
print(f"  Overall Pass Rate             : {summary['overall_pass_rate'] * 100:.1f}%")
print(f"  Policy Rejection Accuracy     : {summary['policy_rejection_accuracy'] * 100:.1f}%")
print(f"  Validator Rejection Accuracy  : {summary['validator_rejection_accuracy'] * 100:.1f}%")
print(f"  Confidence Cap Enforced       : {summary['confidence_cap_enforced']}")